# Manual targeted dataset privacy repair
Run on Spark compute, first DEV, validate, then PROD. This notebook is never a daily/monthly job task. Default is read-only DRY_RUN.
Before APPLY: pause ingestion/promotion jobs and dashboard reads for BOTH environments; publish the verified clean focus/monthly and focus/daily copies to the same GCS paths. Do not upload reference templates to the public/dashboard source. The source preflight refuses unchanged dirty RAW files.
After pushing the generator changes, create/update a separate Git folder from https://github.com/dproiectum/finops-data-generator.git. POLICY_FILE points to its src/finops_generator/privacy.py. Adjust the path to the actual Git folder name. Reuse this authoritative code instead of duplicating sanitization rules in the platform. Never upload original reference datasets. Its checksum is verified before import.
The migration preserves source/fact financial values, ingestion lineage, application codes and entitlements. It repairs business text and OPS business_scope.application_name, canonically rekeys tags and their bridge, and refreshes datamarts. It prints previous Delta versions for manual recovery. The temporary clean bridge is dropped only after PASS.
Current snapshots only: old Delta/GCS versions and local originals still require restricted access and a separate retention decision. No VACUUM or catalog deletion. If anything fails after APPLY, keep jobs and dashboard paused; preserve the output and do not restore one side of a tag relationship alone.
After DEV and PROD PASS: rerun security/05_validate_dashboard_serving_view.sql, restart/redeploy the dashboard, and retest Owner A, Owner B, FinOps Admin and No Access. No billing/daily reload is required.

In [ ]:
from pathlib import Path
import sys
source_root = next((root / 'src' for root in (Path.cwd(), *Path.cwd().parents) if (root / 'src/finops_cloud').is_dir()), None)
if source_root is None:
    raise ValueError('Open this notebook from the updated cloud platform Git folder.')
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

In [ ]:
ENVIRONMENT = 'dev'
CONFIRMATION = ''  # Read-only plan. APPLY requires APPLY_DATASET_PRIVACY_REPAIR.
POLICY_FILE = '/Workspace/Users/thailongdam@gmail.com/finops-data-generator/src/finops_generator/privacy.py'
EXPECTED_POLICY_SHA256 = '19db0d9bdc385171b22dac73634683fc62cab9a64a729c88c97fdf69325d35ee'

In [ ]:
import hashlib
import importlib.util
import json
policy_path = Path(POLICY_FILE)
if not policy_path.is_file():
    raise ValueError('Create/pull the generator Git folder and set POLICY_FILE to its actual privacy.py path. No repair executed.')
if hashlib.sha256(policy_path.read_bytes()).hexdigest() != EXPECTED_POLICY_SHA256:
    raise ValueError('Wrong sanitizer file/version; upload the exact reviewed privacy.py. No repair executed.')
spec = importlib.util.spec_from_file_location('finops_generator_privacy_policy', policy_path)
privacy_policy = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = privacy_policy
spec.loader.exec_module(privacy_policy)
from finops_cloud.config import load_config
from finops_cloud.medallion.privacy_repair import run
result = run(spark, load_config(ENVIRONMENT), privacy_policy, CONFIRMATION)
print(json.dumps(result, indent=2, default=str))